In [126]:
import torch

torch.manual_seed(2026)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device

device(type='cuda')

In [127]:
import numpy as np
import pandas as pd

df_train = pd.read_csv("train_labels.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,image_id,n_dice,total_sum
0,IMG_20191208_111702,10,30
1,IMG_20191208_111816,6,23
2,IMG_20191208_111820,6,24
3,IMG_20191208_111826,6,20
4,IMG_20191208_111833,6,27


In [128]:
img_size = 256
batch_size = 32
epochs = 15
lr = 0.001
n_val = 30

In [142]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

transform = T.Compose([
    T.Resize((img_size, img_size)),
    T.ToTensor()
])

class MyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = f'train/{self.df.image_id.iloc[idx]}.jpg'
        img = self.transform(Image.open(path))
        label1 = torch.tensor(self.df.n_dice.iloc[idx], dtype=torch.float).view(1)
        label2 = torch.tensor(self.df.total_sum.iloc[idx], dtype=torch.float).view(1)
        return img, label1, label2

from sklearn.model_selection import train_test_split
df_1, df_2 = train_test_split(df_train, test_size=0.2, random_state=42)

train_ds = MyDataset(df_1, transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

test_ds = MyDataset(df_2, transform)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

train_ds[0][0][0][0][0]

tensor(0.4353)

In [143]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Flatten(),
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

model = CNN().to(device)
loss_fn = nn.MSELoss()
optim = torch.optim.AdamW(model.parameters(), lr=lr)

model

CNN(
  (net): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): LazyLinear(in_features=0, out_features=64, bias=True)
    (11): ReLU()
    (12): Linear(in_features=64, out_features=32, bias=True)
    (13): ReLU()
    (14): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [144]:
def train(epoch, target):
    model.train()
    total_loss = 0
    for inputs, label1, label2 in train_loader:
        inputs, label1, label2 = inputs.to(device), label1.to(device), label2.to(device)
        outputs = model(inputs)
        if target == 1:
            loss = loss_fn(outputs, label1)
        else:
            loss = loss_fn(outputs, label2)
        optim.zero_grad()
        loss.backward()
        optim.step()
        total_loss += loss.item()
    print("Epoch", epoch)
    print("Training Loss", total_loss/len(train_loader))

def test(epoch, target):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, label1, label2 in test_loader:
            inputs, label1, label2 = inputs.to(device), label1.to(device), label2.to(device)
            outputs = model(inputs)
            if target == 1:
                loss = loss_fn(outputs, label1)
            else:
                loss = loss_fn(outputs, label2)
            total_loss += loss.item()
    print("Validation Loss", total_loss/len(test_loader))

In [145]:
for i in range(1, epochs+1):
    train(i, target=2)
    test(i, target=2)

Epoch 1
Training Loss 271.6261749267578
Validation Loss 74.02621459960938
Epoch 2
Training Loss 101.67887573242187
Validation Loss 65.83184814453125
Epoch 3
Training Loss 77.60504150390625
Validation Loss 65.5560302734375
Epoch 4
Training Loss 63.4618537902832
Validation Loss 87.76265716552734
Epoch 5
Training Loss 65.839794921875
Validation Loss 63.72883605957031
Epoch 6
Training Loss 64.28915634155274
Validation Loss 64.03460884094238
Epoch 7
Training Loss 61.3923843383789
Validation Loss 70.66072273254395
Epoch 8
Training Loss 62.188551330566405
Validation Loss 63.235280990600586
Epoch 9
Training Loss 62.16381607055664
Validation Loss 67.94375991821289
Epoch 10
Training Loss 60.259680938720706
Validation Loss 63.530128479003906
Epoch 11
Training Loss 59.66267547607422
Validation Loss 65.96983337402344
Epoch 12
Training Loss 60.232286834716795
Validation Loss 65.15795516967773
Epoch 13
Training Loss 60.78575592041015
Validation Loss 64.32440376281738
Epoch 14
Training Loss 59.3166419

In [146]:
final_transform = T.Compose([
    T.Resize((img_size, img_size)),
    T.ToTensor()
])

class FinalDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = f'test/{self.df.image_id.iloc[idx]}.jpg'
        img = self.transform(Image.open(path))
        return img

final_ds = FinalDataset(df_test, final_transform)
final_loader = DataLoader(final_ds, batch_size=batch_size)

final_ds[0][0][0][0]

tensor(0.7843)

In [147]:
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        outputs = model(inputs).cpu().numpy().squeeze()
        preds.extend(outputs)

preds = np.array(preds).astype("int")
preds[:5]

array([18, 18, 18, 18, 18])

In [148]:
df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.image_id,
    "answer": preds
})

df_1.head()

,subtaskID,datapointID,answer
0,1,IMG_20191208_111228,18
1,1,IMG_20191208_111246,18
2,1,IMG_20191208_111304,18
3,1,IMG_20191208_111316,18
4,1,IMG_20191208_111328,18


In [149]:
df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.image_id,
    "answer": preds
})

df_2.head()

,subtaskID,datapointID,answer
0,2,IMG_20191208_111228,18
1,2,IMG_20191208_111246,18
2,2,IMG_20191208_111304,18
3,2,IMG_20191208_111316,18
4,2,IMG_20191208_111328,18


In [150]:
submission = pd.concat([df_1, df_2])
submission

,subtaskID,datapointID,answer
0,1,IMG_20191208_111228,18
1,1,IMG_20191208_111246,18
2,1,IMG_20191208_111304,18
3,1,IMG_20191208_111316,18
4,1,IMG_20191208_111328,18
...,...,...,...
45,2,IMG_20191209_095431,17
46,2,IMG_20191209_095439,17
47,2,IMG_20191209_095447,17
48,2,IMG_20191209_095453,17


In [151]:
submission.to_csv("submission.csv", index=False)